# Two-Doer Bottleneck: Training

Two-phase curriculum:
1. **Phase 1 — Pick Object**: Doers start at goal positions, learn object selection from the Seer's message. No navigation.
2. **Phase 2 — Full Training**: Doers start randomly, must navigate the bottleneck corridor then select the correct item.

Phase transition triggers automatically when Phase 1 success exceeds the threshold.

## 1. Install Dependencies

In [ ]:
!pip install -q 'jax[cuda12]' flax optax chex pillow numpy wandb huggingface_hub
!pip install -q navix

## 2. Imports

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import jax
import jax.numpy as jnp
import optax
import flax.linen as nn
from flax.training.train_state import TrainState

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    wandb = None
    WANDB_AVAILABLE = False

# Ensure repo root is on path (adjust if running from a different working directory)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.seer import Seer
from models.doer import Doer
from envs.two_doer_grid import TwoDoerBottleneckEnv, UNSET_TWO_DOER_POSITIONS
from training.loop import generate_two_doer_trajectory_and_gae, make_two_doer_rollout_step
from agents.mappo import update_actor_two_doer, update_critic_two_doer
from eval.metrics import compute_two_doer_cic

print(f"JAX backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")


## 3. Configuration

In [ ]:
config = {
    # --- Core ---
    "seed": 42,
    "use_wandb": False,           # Set True to enable W&B logging
    "wandb_project": "two-doers",
    "wandb_entity": None,

    # --- Architecture ---
    "fsq_levels": [2, 2, 2, 2],   # 4-bit FSQ -> 16 possible messages
    "hidden_size": 128,

    # --- Environment ---
    "grid_height": 10,
    "grid_width": 12,
    "local_view_size": 3,
    "corridor_length": 3,
    "episode_max_steps": 48,
    "doer_perception_level": 2,   # 1=local vision+coords, 2=no vision+coords, 3=blind

    # --- Rewards ---
    "goal_reward": 2.0,
    "progress_reward_scale": 0.1,
    "wrong_selection_penalty": 0.15,
    "wrong_selection_penalty_after_first": 0.30,
    "step_penalty": 0.03,
    "wall_penalty": 0.02,
    "collision_penalty": 0.05,

    # --- Curriculum ---
    "use_pick_object_curriculum": True,
    "two_doer_selection_level_start": 1,          # Start with pick-object phase
    "two_doer_selection_level_advance_threshold": 0.90,  # Advance when success > 90%
    "two_doer_max_selection_attempts": 4,
    "pick_object_max_steps": 8,
    "pick_object_listen_steps": 1,

    # --- Training ---
    "learning_rate": 3e-4,
    "num_envs": 16,
    "num_steps": 64,
    "total_timesteps": 30_000_000,
    "seer_entropy_coef": 0.05,

    # --- Logging / Checkpointing ---
    "log_every": 10,
    "checkpoint_every": 1000,
    "checkpoint_dir": "checkpoints",
    "visualize_dir": "artifacts/episodes",
}

Path(config["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)
Path(config["visualize_dir"]).mkdir(parents=True, exist_ok=True)
print("Config ready.")


## 4. W&B (optional)

In [ ]:
if config["use_wandb"] and WANDB_AVAILABLE:
    wandb.init(project=config["wandb_project"], entity=config["wandb_entity"], config=config)
    print("W&B run:", wandb.run.url)
elif config["use_wandb"]:
    raise ImportError("wandb not installed. Run `pip install wandb` or set use_wandb=False.")
else:
    print("W&B disabled.")


## 5. Helpers

In [ ]:
class GlobalCritic(nn.Module):
    """CTDE critic: observes the full global map to compute a value baseline."""
    output_dim: int = 1

    @nn.compact
    def __call__(self, global_map: jnp.ndarray) -> jnp.ndarray:
        x = nn.Conv(features=32, kernel_size=(3, 3))(global_map)
        x = nn.relu(x)
        x = x.reshape((x.shape[0], -1))
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        return nn.Dense(features=self.output_dim)(x)


def initialize_two_doer_carry(doer, num_envs, num_doers, hidden_size):
    flat = doer.initialize_carry(batch_size=num_envs * num_doers, hidden_size=hidden_size)
    return jax.tree_util.tree_map(lambda x: x.reshape((num_envs, num_doers, hidden_size)), flat)


def get_active_message_levels(fsq_levels, active_bits):
    return [fsq_levels[i] if i < active_bits else 1 for i in range(len(fsq_levels))]


def compute_message_stats(messages, active_levels):
    codes = np.asarray(messages).reshape(-1, messages.shape[-1])
    strs = [str(list(r)) for r in codes]
    counter = Counter(strs)
    total = len(strs)
    probs = np.array([c / total for c in counter.values()])
    entropy = float(-np.sum(probs * np.log(probs + 1e-8)))
    max_h = float(np.log(np.prod(active_levels)))
    return {
        "rollout_message_entropy_normalized": entropy / max_h if max_h > 0 else 0.0,
        "rollout_message_unique_codes": len(counter),
        "rollout_message_num_codes": int(np.prod(active_levels)),
    }


def save_checkpoint(params, step, label="checkpoint", checkpoint_dir="checkpoints"):
    from flax.training import checkpoints
    path = checkpoints.save_checkpoint(
        ckpt_dir=str(checkpoint_dir), target=params, step=int(step), overwrite=True,
    )
    print(f"[{label}] Saved -> {path}")
    return path


print("Helpers defined.")


## 6. Initialize Environment and Models

In [ ]:
rng = jax.random.PRNGKey(config["seed"])
rng, seer_rng, doer_rng, critic_rng = jax.random.split(rng, 4)

initial_phase = config["two_doer_selection_level_start"] if config["use_pick_object_curriculum"] else 2

env = TwoDoerBottleneckEnv(
    grid_height=config["grid_height"],
    grid_width=config["grid_width"],
    local_view_size=config["local_view_size"],
    corridor_length=config["corridor_length"],
    max_steps=config["episode_max_steps"],
    progress_reward_scale=config["progress_reward_scale"],
    goal_reward=config["goal_reward"],
    wrong_selection_penalty=config["wrong_selection_penalty"],
    wrong_selection_penalty_after_first=config["wrong_selection_penalty_after_first"],
    step_penalty=config["step_penalty"],
    wall_penalty=config["wall_penalty"],
    collision_penalty=config["collision_penalty"],
    doer_perception_level=config["doer_perception_level"],
    selection_phase_level=initial_phase,
    max_selection_attempts=config["two_doer_max_selection_attempts"],
    pick_object_max_steps=config["pick_object_max_steps"],
    pick_object_listen_steps=config["pick_object_listen_steps"],
)
print(f"Phase: {env.phase_name} (level={env.selection_phase_level})")
print(f"Doer perception level: {env.doer_perception_level}")
print(f"Actions: {env.num_actions} | Doers: {env.num_doers}")


In [ ]:
rng, reset_rng = jax.random.split(rng)
env_obs, env_state = env.reset_batch(jax.random.split(reset_rng, config["num_envs"]))

fsq_levels = config["fsq_levels"]
seer   = Seer(fsq_levels=fsq_levels, num_actions=env.num_actions, num_message_heads=env.num_doers)
doer   = Doer(fsq_levels=fsq_levels, num_actions=env.num_actions)
critic = GlobalCritic(output_dim=1)

dummy_map    = env_obs["global_map"][:1]
dummy_sym    = env_obs["symbolic_state"][:1]
dummy_local  = env_obs["local_views"][:1, 0]
dummy_prop   = env_obs["proprioceptions"][:1, 0]
dummy_target = env_obs["target_images"][:1]
dummy_menu   = env_obs["menu_images"][:1, 0]
dummy_msg    = jnp.zeros((1, len(fsq_levels)))
sc = seer.initialize_carry(1, config["hidden_size"])
dc = doer.initialize_carry(1, config["hidden_size"])

seer_params   = seer.init(seer_rng,   sc, dummy_map, dummy_sym, dummy_target)["params"]
doer_params   = doer.init(doer_rng,   dc, dummy_local, dummy_prop, dummy_msg, dummy_menu)["params"]
critic_params = critic.init(critic_rng, dummy_map)["params"]
params = {"seer": seer_params, "doer": doer_params, "critic": critic_params}

tx = optax.chain(optax.clip_by_global_norm(0.5), optax.adam(config["learning_rate"], eps=1e-5))
actor_state  = TrainState.create(apply_fn=None, params={"seer": seer_params, "doer": doer_params}, tx=tx)
critic_state = TrainState.create(apply_fn=critic.apply, params=critic_params, tx=tx)

seer_carry = seer.initialize_carry(config["num_envs"], config["hidden_size"])
doer_carry = initialize_two_doer_carry(doer, config["num_envs"], env.num_doers, config["hidden_size"])
step_fn    = make_two_doer_rollout_step(env, seer.apply, doer.apply, critic.apply)

num_updates = config["total_timesteps"] // (config["num_steps"] * config["num_envs"])
print(f"Models initialized. Total updates: {num_updates:,}")


## 7. Training Loop

In [ ]:
phase1_checkpoint_saved = False

for update in range(num_updates):
    rng, rollout_rng = jax.random.split(rng)
    init_seer_carry = seer_carry
    init_doer_carry = doer_carry

    # Collect trajectory + GAE
    final_runner_state, trajectory_batch = generate_two_doer_trajectory_and_gae(
        params, rollout_rng, env_obs, env_state,
        seer_carry, doer_carry, UNSET_TWO_DOER_POSITIONS,
        config["num_steps"], step_fn, critic.apply,
    )
    params, seer_carry, doer_carry, env_state, env_obs, _, _ = final_runner_state

    # Policy and value updates
    rng, actor_rng, critic_rng_key = jax.random.split(rng, 3)
    batched = jax.tree_util.tree_map(lambda x: jnp.swapaxes(x, 0, 1), trajectory_batch)
    active_msg_levels = get_active_message_levels(fsq_levels, env.active_message_bits)

    actor_state, actor_metrics = update_actor_two_doer(
        actor_state, batched, init_seer_carry, init_doer_carry,
        seer.apply, doer.apply, actor_rng,
        active_msg_levels, env.active_message_bits, env.is_pick_object_phase,
        jnp.asarray(config["seer_entropy_coef"], dtype=jnp.float32),
    )
    critic_state, critic_metrics = update_critic_two_doer(
        critic_state, batched, critic.apply, critic_rng_key,
    )
    params["seer"]   = actor_state.params["seer"]
    params["doer"]   = actor_state.params["doer"]
    params["critic"] = critic_state.params

    # Compute metrics
    done_mask      = trajectory_batch.done
    n_episodes     = max(1, int(done_mask.sum()))
    first_try_rate = float(jnp.logical_and(done_mask, trajectory_batch.first_try_success).sum()) / n_episodes
    eventual_rate  = float(jnp.logical_and(done_mask, trajectory_batch.eventual_success).sum()) / n_episodes
    total_sel      = float(trajectory_batch.valid_selection_count.sum())
    correct_sel    = float(trajectory_batch.correct_selection_count.sum())
    sel_acc        = correct_sel / max(1.0, total_sel)
    msg_stats      = compute_message_stats(trajectory_batch.message[..., :env.active_message_bits], active_msg_levels)

    # Logging
    if update % config["log_every"] == 0:
        global_step = (update + 1) * config["num_steps"] * config["num_envs"]
        print(
            f"[{update:5d}/{num_updates}] Phase={env.phase_name:20s} | "
            f"Success={first_try_rate:.3f} | Eventual={eventual_rate:.3f} | "
            f"SelAcc={sel_acc:.3f} | "
            f"MsgH={msg_stats['rollout_message_entropy_normalized']:.3f} | "
            f"MsgUsed={msg_stats['rollout_message_unique_codes']}/{msg_stats['rollout_message_num_codes']} | "
            f"DoerGrad={actor_metrics.get('doer_grad_norm', 0.0):.4f}"
        )
        if config["use_wandb"] and WANDB_AVAILABLE:
            wandb.log({
                "phase": env.selection_phase_level,
                "first_try_success_rate": first_try_rate,
                "eventual_success_rate": eventual_rate,
                "selection_accuracy": sel_acc,
                "message_entropy_normalized": msg_stats["rollout_message_entropy_normalized"],
                "message_unique_codes": msg_stats["rollout_message_unique_codes"],
                "team_reward": float(trajectory_batch.reward.mean()),
                "seer_grad_norm": actor_metrics.get("seer_grad_norm", 0.0),
                "doer_grad_norm": actor_metrics.get("doer_grad_norm", 0.0),
                "critic_loss": critic_metrics.get("critic_loss", 0.0),
            }, step=global_step)

    # Phase 1 -> Phase 2 transition
    if (
        config["use_pick_object_curriculum"]
        and env.selection_phase_level == 1
        and first_try_rate > config["two_doer_selection_level_advance_threshold"]
    ):
        global_step = (update + 1) * config["num_steps"] * config["num_envs"]
        print("\n" + "=" * 60)
        print("  PHASE TRANSITION: Pick-Object -> Full Training")
        print(f"  Success={first_try_rate:.3f} > threshold={config['two_doer_selection_level_advance_threshold']}")
        print("=" * 60 + "\n")

        if not phase1_checkpoint_saved:
            save_checkpoint(params, global_step, label="phase1_complete", checkpoint_dir=config["checkpoint_dir"])
            phase1_checkpoint_saved = True

        env.set_selection_phase_level(2)
        step_fn = make_two_doer_rollout_step(env, seer.apply, doer.apply, critic.apply)

        rng, reset_rng = jax.random.split(rng)
        env_obs, env_state = env.reset_batch(jax.random.split(reset_rng, config["num_envs"]))
        seer_carry = seer.initialize_carry(config["num_envs"], config["hidden_size"])
        doer_carry = initialize_two_doer_carry(doer, config["num_envs"], env.num_doers, config["hidden_size"])

    # Periodic checkpoint
    if update > 0 and update % config["checkpoint_every"] == 0:
        global_step = (update + 1) * config["num_steps"] * config["num_envs"]
        save_checkpoint(params, global_step, label=f"update_{update}", checkpoint_dir=config["checkpoint_dir"])

print("Training complete.")


## 8. Save Final Checkpoint

In [ ]:
final_step = num_updates * config["num_steps"] * config["num_envs"]
save_checkpoint(params, final_step, label="final", checkpoint_dir=config["checkpoint_dir"])
if config["use_wandb"] and WANDB_AVAILABLE:
    wandb.finish()
